# Advanced aggregation

In [ ]:
Topics:

ROLLUP
CUBE
GROUPING SETS

These help generate multiple summary reports in a single query.

In [ ]:
The Problem

Suppose we have:
| region | department | revenue |
| ------ | ---------- | ------- |
| North  | IT         | 100     |
| North  | HR         | 50      |
| South  | IT         | 200     |
| South  | HR         | 80      |

Normal query:

SELECT
    region,
    department,
    SUM(revenue)
FROM sales
GROUP BY region, department;

Result:
| region | department | revenue |
| ------ | ---------- | ------- |
| North  | IT         | 100     |
| North  | HR         | 50      |
| South  | IT         | 200     |
| South  | HR         | 80      |



### ROLLUP()

In [ ]:
Think:

"Give me details + subtotals + grand total."

Query
SELECT
    region,
    department,
    SUM(revenue) AS revenue
FROM sales
GROUP BY ROLLUP(region, department);

Result
| region | department | revenue |
| ------ | ---------- | ------- |
| North  | IT         | 100     |
| North  | HR         | 50      |
| North  | NULL       | 150     |
| South  | IT         | 200     |
| South  | HR         | 80      |
| South  | NULL       | 280     |
| NULL   | NULL       | 430     |

Interpretation
North IT     = 100
North HR     = 50
North Total  = 150

South IT     = 200
South HR     = 80
South Total  = 280

Grand Total  = 430


Mental Model
ROLLUP(region, department)

creates:

(region, department)
(region)
()

Where:

()

means Grand Total.


In [ ]:
Why BI Teams Love ROLLUP

Instead of running:

GROUP BY region, department

then

GROUP BY region

then

SUM(revenue)

you run one query.

### CUBE()

In [ ]:
This is stronger.

Think:

"Give me ALL possible combinations."

Query
SELECT
    region,
    department,
    SUM(revenue)
FROM sales
GROUP BY CUBE(region, department);


Result

Regular groups:
| region | department |
| ------ | ---------- |
| North  | IT         |
| North  | HR         |
| South  | IT         |
| South  | HR         |

Region totals:
| region | department |
| ------ | ---------- |
| North  | NULL       |
| South  | NULL       |


Department totals:
| region | department |
| ------ | ---------- |
| NULL   | IT         |
| NULL   | HR         |


Grand Total:
| region | department |
| ------ | ---------- |
| NULL   | NULL       |



In [ ]:
Difference

ROLLUP

Details
→ Region Totals
→ Grand Total

Hierarchy.

CUBE
Everything possible

All combinations.

### GROUPING SETS()

In [ ]:
Most flexible.

You specify exactly what summaries you want.

Suppose management wants:

Region totals
Department totals
Region + Department totals

Query:

SELECT
    region,
    department,
    SUM(revenue)
FROM sales
GROUP BY GROUPING SETS (
    (region, department),
    (region),
    (department)
);

Now PostgreSQL produces only those summaries.

No grand total unless you explicitly ask.

### Real-World Example

In [ ]:
Suppose you're building a Power BI dashboard.

Management wants:

Revenue by:
Region
Department
Region + Department
Entire Company

Instead of 4 queries:

GROUP BY region
GROUP BY department
GROUP BY region, department
SUM(revenue)

you can use:

CUBE()

or

GROUPING SETS()